In [1]:
import pandas as pd
from transformers import pipeline
import torch

torch.cuda.is_available()

c:\Users\Kaylen\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 22:47:07.423000 38240 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


True

In [ ]:
# Read in my data (parquet file)
posts_data = pd.read_parquet("data/clean_posts.parquet") # text is in 'full_text' column
posts_data.rename(columns={'full_text': 'document_text', 'link_flair_text': 'post_flair'}, inplace=True)
posts_data['document_id'] = posts_data['post_id']
posts_data['document_type'] = "post"
posts_data['link'] = "https://www.reddit.com/comments/" + posts_data["post_id"]

comments_data = pd.read_parquet("data/clean_comments.parquet") # text is in 'body' column
comments_data.rename(columns={'body': 'document_text'}, inplace=True)
comments_data['document_id'] = comments_data['comment_id']
comments_data['link'] = "https://www.reddit.com" + comments_data['permalink']
comments_data['document_type'] = "comment"

# Assigning comment records a post_flair value based on the post they belong to.
comments_data = comments_data.merge(
    posts_data[["post_id", "post_flair"]],  # right table: id and post_flair (additional column after join)
    on="post_id", how="left", validate="many_to_one"
)

common_columns = list(set(posts_data.columns).intersection(set(comments_data.columns)))
keep_columns_and_order = [
    "document_id",
    "document_type",
    "post_id",
    "parent_id",
    "created_utc",
    "post_flair",
    "author",
    "mains",
    "document_text",
    "score",
    "num_comments",
    "link",
    "subreddit",
]

# Append posts + comments into one table
full_data = pd.concat(
    [posts_data.reindex(columns=keep_columns_and_order),
    comments_data.reindex(columns=keep_columns_and_order)],
    ignore_index=True,
)

full_data["num_comments"] = full_data["num_comments"].astype("Int64")  # keep ints; NA for comments

# Post / top-level comment / reply
full_data["reply_level"] = "post"
full_data.loc[full_data["parent_id"].str.startswith("t3_", na=False), "reply_level"] = "top-level comment"
full_data.loc[full_data["parent_id"].str.startswith("t1_", na=False), "reply_level"] = "reply"


In [3]:
full_data["word_count"] = full_data["document_text"].str.split().str.len()

short_docs = full_data[full_data["word_count"] <= 5]
print(len(short_docs), "short docs")

# A random sample of 30 (re-run for a different sample)
short_docs[["document_type", "post_flair", "document_text", "link"]].sample(30)

6507 short docs


,document_type,post_flair,document_text,link
24558,comment,Cosplay,:two_hearts: :two_hearts: :two_hearts:,https://www.reddit.com/r/LoveAndDeepspace/comm...
29783,comment,Valko,\#truthnuke,https://www.reddit.com/r/LoveAndDeepspace/comm...
32236,comment,Discussion,Same unfortunately :loudly_crying_face:,https://www.reddit.com/r/LoveAndDeepspace/comm...
27935,comment,Discussion,There’ll be Valko! :face_with_steam_from_nose:,https://www.reddit.com/r/LoveAndDeepspace/comm...
10993,comment,Discussion,:handshake: :handshake: :handshake:,https://www.reddit.com/r/LoveAndDeepspace/comm...
51786,comment,Official Media,:face_with_tears_of_joy: True!,https://www.reddit.com/r/LoveAndDeepspace/comm...
1378,post,Valko,Valko come back home!,https://www.reddit.com/comments/1uosvxn
2188,post,Edits,Valko Billboard in Milan,https://www.reddit.com/comments/1wpta61
22530,comment,Art,#bringvalkoback :loudly_crying_face: :loudly_c...,https://www.reddit.com/r/LoveAndDeepspace/comm...
36979,comment,Art,Thank youuuu :heart_hands_light_skin_tone: :fo...,https://www.reddit.com/r/LoveAndDeepspace/comm...


In [ ]:
# There are multiple names for the company and love interests. It's best to normalize them to a single name for each so that they're treated as one entity for topic modeling.
to_normalize = {
    # Game / company
    r"(?i)love\s*(?:and|&)\s*deep\s*space": "LADS",
    r"(?i)\bpaper\s*games\b": "Infold",

    # Love interests: alt names -> English name
    r"(?i)\bshen\s*xing\s*hui\b|\bseiya\b|\bshim\s*seong\s*hun\b|沈星回": "Xavier",
    r"(?i)\bli\s*shen\b|\blee\s*seo\s*eon\b|黎深": "Zayne",
    r"(?i)\bqi\s*yu\b|\bhomura\b|\bgi\s*wook\b|祁煜": "Rafayel",
    r"(?i)\bqin\s*che\b|秦[彻徹]|シン|\bjin\s*woon\b|진운": "Sylus",
    r"(?i)\bxia\s*yi\s*zhou\b|夏以昼|\bmahiru\b|マヒル|\bha\s*woo\s*joo\b|하우주": "Caleb",
    r"(?i)\bao\s*yin\b|敖尹": "Valko"
}

In [ ]:
# Lookalike letters affected two text documents. To noramlize them, we can replace the lookalike letters with their English counterparts.
lookalike_map = str.maketrans({
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ѕ": "s",
    "ј": "j", "һ": "h", "ԁ": "d", "ԝ": "w", "ԛ": "q", "м": "m", "т": "t", "н": "h", "к": "k",
    "Н": "H", "Р": "P", "С": "C", "Ԝ": "W",
    "ո": "n", "ս": "u", "ⅼ": "l", "ⅿ": "m", "ɡ": "g", "ν": "v", "ι": "i", "Ι": "I", "Ν": "N",
    "ᖯ": "b", "ꓓ": "D", "ꓝ": "F", "ꓣ": "R", "𝈖": "r",
})

# Only touches docs using these characters.
disguised = full_data["document_text"].str.contains("[ոսԁԝᖯ]")
full_data.loc[disguised, "document_text"] = full_data.loc[disguised, "document_text"].str.translate(lookalike_map)


In [ ]:
# Flag non-English, using the ORIGINAL text
english_chars = full_data["document_text"].str.count(r"[A-Za-z]")
foreign_chars = full_data["document_text"].str.count(
    r"[\u4e00-\u9fff\u3040-\u309f\uac00-\ud7af\u0400-\u04ff]"  # Chinese, Japanese, Korean, Cyrillic
)
full_data["is_english"] = foreign_chars <= english_chars

# Create normalized text for the models
full_data["nlp_text"] = full_data["document_text"]
for pattern, replacement in to_normalize.items():
    full_data["nlp_text"] = full_data["nlp_text"].str.replace(pattern, replacement, regex=True)

# Only English docs for the models
nlp_docs = full_data[full_data["is_english"]]

In [ ]:
emotion_pipeline = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions", device = 0)
#results = emotion_pipeline(texts, batch_size=32, truncation=True)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1537.63it/s]
